# 🏗️ Deepfake Audio Detection — ResNet50
**Notebook:** `04_resnet50.ipynb`  
**Team:** Kazim Mammadli · Vidadi Javadov · Zamin Sultanli  
**Purpose:** Transfer-learning experiment using ResNet50 pretrained on ImageNet. Two-phase training: frozen base → fine-tune last 30 layers.

### Training phases
| Phase | Base layers | Epochs | LR | Notes |
|-------|------------|--------|-----|-------|
| 1 | Frozen | 10 | 1e-3 | Train classification head only |
| 2 | Last 30 unfrozen | 10 | 1e-5 | Fine-tune with low LR |

---
## 1. Environment & Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import numpy as np
import tensorflow as tf
from tensorflow import keras

# Pipeline
from src.features.mel_spectrogram import extract, set_preset, _cfg
from src.data.dataset import FoRDataset, build_tf_datasets

# Shared utilities (no inline duplication)
from src.training.preprocessing import preprocess_for_model
from src.evaluation.visualize import plot_history
from src.evaluation.metrics import evaluate_model

set_preset('for-2sec')

print(f'Python     : {sys.version.split()[0]}')
print(f'TensorFlow : {tf.__version__}')
print(f'Mel config : {_cfg}')
print(f'GPU        : {tf.config.list_physical_devices("GPU")}')
print()
print('✅ All imports successful')

---
## 2. Load TF Datasets

In [ ]:
DATASET_DIR = '../data/raw/for-2seconds'
BATCH_SIZE  = 32

print('Building TF datasets...')
train_ds, val_ds, test_ds = build_tf_datasets(
    DATASET_DIR, preset='for-2sec', batch_size=BATCH_SIZE
)

for x, y in train_ds.take(1):
    print(f'Raw batch — x: {x.shape}, dtype={x.dtype} | y: {y.shape}, dtype={y.dtype}')
print('\n✅ Datasets loaded')

---
## 3. Apply Preprocessing Adapter — ResNet50 Mode

`preprocess_for_model` is defined once in `src/training/preprocessing.py` and imported here.  
Pipeline output `(batch, 1, 80, target_frames)` → ResNet50 input `(batch, 224, 224, 3)`:
1. Transpose to channels-last
2. Repeat channel dim → pseudo-RGB
3. Resize to `(224, 224)`
4. `resnet50.preprocess_input` → zero-centred BGR

In [ ]:
# preprocess_for_model is imported from src.training.preprocessing
train_resnet = train_ds.map(
    lambda x, y: preprocess_for_model(x, y, 'resnet50'),
    num_parallel_calls=tf.data.AUTOTUNE
)
val_resnet = val_ds.map(
    lambda x, y: preprocess_for_model(x, y, 'resnet50'),
    num_parallel_calls=tf.data.AUTOTUNE
)
test_resnet = test_ds.map(
    lambda x, y: preprocess_for_model(x, y, 'resnet50'),
    num_parallel_calls=tf.data.AUTOTUNE
)

for x, y in train_resnet.take(1):
    print(f'ResNet50 batch — x: {x.shape}   (expected: ({BATCH_SIZE}, 224, 224, 3))')
    print(f'                 y: {y.shape}')
print('\n✅ Preprocessing adapter applied (resnet50 mode)')

---
## 4. Build ResNet50 Model

In [ ]:
def build_resnet50():
    """ResNet50 + custom classification head for binary deepfake detection."""
    base_model = keras.applications.ResNet50(
        input_shape=(224, 224, 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False  # Phase 1: frozen

    inputs  = keras.Input(shape=(224, 224, 3), name='image_input')
    x       = base_model(inputs, training=False)
    x       = keras.layers.GlobalAveragePooling2D(name='gap')(x)
    x       = keras.layers.Dropout(0.3, name='dropout')(x)
    outputs = keras.layers.Dense(1, activation='sigmoid', name='output')(x)

    model = keras.Model(inputs, outputs, name='resnet50_deepfake')
    return model, base_model


model, base_model = build_resnet50()

print(f'Base model layers : {len(base_model.layers)}')
print(f'Total params      : {model.count_params():,}')
print(f'Trainable params  : {sum(tf.size(v).numpy() for v in model.trainable_variables):,}  (head only)')
print(f'Non-trainable     : {sum(tf.size(v).numpy() for v in model.non_trainable_variables):,}  (frozen base)')

---
## 5. Phase 1 — Frozen Base (Train Head Only)

In [ ]:
os.makedirs('../models', exist_ok=True)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks_phase1 = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=3,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath='../models/resnet50_phase1.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
]

print('Phase 1 — Training head (base frozen)...')
history_p1 = model.fit(
    train_resnet,
    validation_data=val_resnet,
    epochs=10,
    callbacks=callbacks_phase1,
    verbose=1
)
print('\n✅ Phase 1 complete')

---
## 6. Phase 1 — Training Curves

`plot_history` is defined once in `src/evaluation/visualize.py` and imported here.

In [ ]:
# plot_history is imported from src.evaluation.visualize
os.makedirs('../outputs', exist_ok=True)
plot_history(
    history_p1,
    title='ResNet50 — Phase 1 (Frozen Base)',
    save_path='../outputs/resnet50_phase1_curves.png'
)

---
## 7. Phase 2 — Fine-Tune Last 30 Layers

In [ ]:
# Unfreeze last 30 layers of the base model
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

trainable_now = sum(tf.size(v).numpy() for v in model.trainable_variables)
print(f'Trainable params after unfreeze: {trainable_now:,}')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks_phase2 = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=3,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath='../models/resnet50.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
]

print('Phase 2 — Fine-tuning last 30 layers (lr=1e-5)...')
history_p2 = model.fit(
    train_resnet,
    validation_data=val_resnet,
    epochs=10,
    callbacks=callbacks_phase2,
    verbose=1
)
print('\n✅ Phase 2 complete')

---
## 8. Phase 2 — Training Curves

In [ ]:
plot_history(
    history_p2,
    title='ResNet50 — Phase 2 (Fine-Tuning Last 30 Layers)',
    save_path='../outputs/resnet50_phase2_curves.png'
)

---
## 9. Evaluate on Test Set

`evaluate_model` is defined once in `src/evaluation/metrics.py` and imported here.

In [ ]:
# evaluate_model is imported from src.evaluation.metrics
results = evaluate_model(
    model,
    test_resnet,
    model_name='ResNet50',
    cmap='Oranges',
    output_dir='../outputs'
)

print(f"\nFinal — Acc: {results['accuracy']:.4f} | "
      f"F1: {results['f1']:.4f} | AUC: {results['auc']:.4f}")